# [Video RAG] Retrieval & Generation

In [ ]:
%pip install -Uq langchain langchain-openai langchain-pinecone

In [ ]:
# 구글드라이브 연동
from google.colab import drive
drive.mount('/content/drive')  # 내 구글드라이브를 /content/drive 경로로 마운트

BASE_PATH = '/content/drive/MyDrive/SK네트웍스 Family AI 캠프 23기/05_multimodal_rag'  # 기본작업경로 변수

In [ ]:
import os
from google.colab import userdata  # Colab 비밀키 저장소 접근 모듈

# 환경변수 설정
os.environ['LANGSMITH_TRACING'] = 'true'  # LangSmith 추적 기능 활성화
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_API_KEY'] = userdata.get('LANGSMITH_API_KEY')
os.environ['LANGSMITH_PROJECT'] = 'skn23-multimodal-rag'
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['PINECONE_API_KEY'] = userdata.get('PINECONE_API_KEY')

## Chain 구성

In [ ]:
from typing import Optional  # Optional 타입 (값이 없을수도 있음)
from pydantic import BaseModel, Field  # 구조화된 응답 스키마(검증 / 설명)
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate

class VideoAnswer(BaseModel):
    """비디오 검색 결과에 대한 답변 구조 클래스"""
    found: bool = Field(description='적절한 비디오 검색결과를 찾았는지 여부')
    answer: str = Field(description='질문에 대해 검색결과 기반의 답변내용')
    video_filename: Optional[str] = Field(description='참조한 비디오 파일명')
    frame_no: Optional[str] = Field(description='참조한 프레임명')
    frame_caption: Optional[str] = Field(description='참조함 프레임설명')


embedding = OpenAIEmbeddings(model='text-embedding-3-small')
vector_store = PineconeVectorStore(
    index_name='video-caption-kr',
    embedding=embedding
)

llm = init_chat_model('openai:gpt-4.1-mini', temperature=0.2)
prompt = PromptTemplate.from_template('''
당신은 비디오 장면 검색 에이젼트입니다.
제공된 비디오 검색결과에서 사용자의 질문에 적합한 장면을 찾고, 답변해주세요.
제공된 비디오 검색결과에서 사용자의 질문에 적합한 장면이 없다면, 검색결과가 없다고 답변해주세요.

[사용자 질문]
{question}

[참조할 비디오 검색결과(프레임 설명)]
{context}
''')
chain = prompt | llm.with_structured_output(VideoAnswer)  # 프롬프트 -> 구조화된 형태로 응답받는 체인

## 헬퍼함수

In [ ]:
import base64
from IPython.display import HTML

# 검색 결과 문서 리스트(docs)를 LLM 입력용 context 문자열로 합치는 함수
def build_context(docs):
    context = ""
    for i, doc in enumerate(docs, 1):  # 문서들 순회 (1부터 i값)
        metadata = doc.metadata
        context += f'''
[{i}]
비디오: {metadata['video_filename']}
프레임: {metadata['frame_no']}
해설: {doc.page_content}
'''
    return context

def get_frame_image_path(video_filename, frame_no, frame_dir='frames'):
    """주어진 경로를 조합해 프레임이미지의 절대경로 반환"""
    video_basename = os.path.splitext(video_filename)[0]  # 파일명
    frame_no = int(float(frame_no)) # str -> float -> int (예: '30.0')
    filename = f'{video_basename}_frame{frame_no:05d}.jpg'  # 파일명 완성
    return os.path.join(BASE_PATH, frame_dir, video_basename, filename)  # BASE_PATH/frames/{video}/{file}

# display_video(): html video태그 생성 함수
def display_video(video_filename, video_dir='videos', width=300):
    video_path = os.path.join(BASE_PATH, video_dir, video_filename)

    with open(video_path, 'rb') as f:
        video_b64 = base64.b64encode(f.read()).decode('utf-8')  # mp4 바이너리 파일을 base64 문자열로 반환

    html = f'''
<video width="{width}" controls>
    <source src="data:video/mp4;base64,{video_b64}"/>
</video>
'''
    display(HTML(html))

# 이미지 파일을 base64로 인코딩해 HTML <img> 태그로 출력하는 함수
def display_image(frame_image_path, width=300):
    with open(frame_image_path, 'rb') as f:
        image_b64 = base64.b64encode(f.read()).decode('utf-8')

    html = f'''
<img src='data:image/jpeg;base64,{image_b64}' width='{width}'/>
'''
    display(HTML(html))


## MultimodalRAG
- RAG Chain
- RAG Agent

In [ ]:
# 사용자 질문을 이용해 유사한 frame을 찾고, LLM이 구조화된 답변을 만든 뒤 결과를 시각화하는 함수
def video_search_and_answer(query, k=3):
    # 1. 벡터스토어 검색
    docs = vector_store.similarity_search(query, k=k)
    # print(docs, end='\n\n')

    # 2. 체인실행 (프롬프트 증강)
    context = build_context(docs)  # 검색된 문서들을 LLM 입력용 context 문자열로 구성
    response = chain.invoke({'question': query, 'context': context})
    # print(response)

    # 3. 응답 VideoAnswer객체를 포맷팅
    if response.found:  # 적절한 장면을 찾으면
        video_filename = response.video_filename  # 참조 비디오파일명
        frame_no = response.frame_no              # 참조 프레임 번호
        print('비디오 검색결과를 찾았습니다...')
        print('[비디오 플레이어]:')
        display_video(video_filename)  # HTML 플레이어로 렌더링
        print('[관련프레임]:')
        display_image(get_frame_image_path(video_filename, frame_no))  # HTML 이미지 렌더링
        print('[비디오 파일명]:')
        print(response.video_filename)
        print('[프레임번호]:')
        print(response.frame_no)
        print('[답변]:')
        print(response.answer)
    else:
        print('검색된 비디오/프레임 정보가 없습니다.')


video_search_and_answer('농구공 드리블하면서 상대 제치는 장면 찾아줘')
# video_search_and_answer('축구공 드리블하는 장면 있어?')
# video_search_and_answer('등산하는 장면 찾아줘')
# video_search_and_answer('설원에서 스키타면서 내려오는 장면')


In [ ]:
video_search_and_answer('축구공 드리블하는 장면 있어?')

In [ ]:
video_search_and_answer('등산')

In [ ]:
video_search_and_answer('설원에서 스키타면서 내려오는 장면')